# ML Prediction API — Test Suite
This notebook mirrors `test_app.py` and runs all tests interactively using FastAPI's `TestClient` — no running server needed.

## 1. Install Dependencies

In [1]:
!pip install fastapi uvicorn joblib scikit-learn numpy pydantic httpx pytest

## 2. Train & Save a Sample Model
Required so `app.py` can load `model.joblib` at import time.

In [2]:
import numpy as np
import joblib
from sklearn.linear_model import LinearRegression

np.random.seed(42)
X_train = np.random.rand(100, 4)
y_train = X_train @ np.array([1.5, -2.0, 3.0, 0.5]) + np.random.randn(100) * 0.1

model = LinearRegression()
model.fit(X_train, y_train)
joblib.dump(model, 'model.joblib')
print(f'Model saved. n_features_in_: {model.n_features_in_}')

Model saved. n_features_in_: 4


## 3. Write `app.py` to Disk

In [3]:
%%writefile app.py
import logging
from pathlib import Path
from typing import List

import joblib
import numpy as np
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, ConfigDict, Field

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

class InputData(BaseModel):
    features: List[float] = Field(..., description='Input features for prediction')
    model_config = ConfigDict(json_schema_extra={'example': {'features': [1.0, 2.0, 3.0, 4.0]}})

class OutputData(BaseModel):
    prediction: float = Field(..., description='Model prediction')

app = FastAPI(title='ML Prediction API', version='1.0.0')

MODEL_PATH = Path(__file__).parent / 'model.joblib'

try:
    model = joblib.load(str(MODEL_PATH))
    logger.info(f'Model loaded from {MODEL_PATH}')
    EXPECTED_FEATURES = model.n_features_in_
except Exception as e:
    logger.error(f'Failed to load model: {e}')
    model = None
    EXPECTED_FEATURES = 4

@app.post('/predict', response_model=OutputData)
async def predict(data: InputData) -> OutputData:
    try:
        if len(data.features) != EXPECTED_FEATURES:
            raise HTTPException(
                status_code=422,
                detail=f'Expected {EXPECTED_FEATURES} features, got {len(data.features)}',
            )
        if model is None:
            raise HTTPException(status_code=500, detail='Model not loaded')
        X = np.array(data.features).reshape(1, -1)
        prediction = model.predict(X)[0]
        return OutputData(prediction=float(prediction))
    except HTTPException:
        raise
    except Exception as e:
        logger.error(f'Prediction error: {e}')
        raise HTTPException(status_code=500, detail=f'Model inference failed: {str(e)}') from e

Overwriting app.py


## 4. Setup — Import App & Create TestClient

In [4]:
import time
from unittest.mock import patch
from fastapi.testclient import TestClient
from app import InputData, app

client = TestClient(app)

# Shared fixtures (as plain dicts/objects for notebook use)
valid_input_data = InputData(features=[1.0, 2.0, 3.0, 4.0])
valid_input_dict = {'features': [1.0, 2.0, 3.0, 4.0]}

print('TestClient ready.')

INFO:app:Model loaded from C:\Users\ryan\model.joblib


TestClient ready.


## 5. Test: Valid Input (Happy Path)

In [5]:
response = client.post('/predict', json=valid_input_dict)

assert response.status_code == 200, f'Expected 200, got {response.status_code}'
assert 'prediction' in response.json(), 'Missing prediction key'
prediction = response.json()['prediction']
assert isinstance(prediction, (int, float)), f'Expected number, got {type(prediction)}'

print('PASS — test_predict_valid_input')
print(f'  Status : {response.status_code}')
print(f'  Response: {response.json()}')

INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


PASS — test_predict_valid_input
  Status : 200
  Response: {'prediction': 8.38553963404306}


## 6. Test: Invalid Input (Wrong Number of Features)

In [6]:
invalid_input = {'features': [1.0, 2.0]}  # Only 2 features instead of 4
response = client.post('/predict', json=invalid_input)

assert response.status_code in [422, 500], f'Expected 422 or 500, got {response.status_code}'

print('PASS — test_predict_invalid_input')
print(f'  Status : {response.status_code}')
print(f'  Detail : {response.json()}')

INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


PASS — test_predict_invalid_input
  Status : 422
  Detail : {'detail': 'Expected 4 features, got 2'}


## 7. Test: Missing Required Field

In [7]:
response = client.post('/predict', json={})

assert response.status_code == 422, f'Expected 422, got {response.status_code}'

print('PASS — test_predict_missing_required_field')
print(f'  Status : {response.status_code}')
print(f'  Detail : {response.json()["detail"][0]["msg"]}')

INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


PASS — test_predict_missing_required_field
  Status : 422
  Detail : Field required


## 8. Test: Wrong Data Type

In [8]:
invalid_input = {'features': 'not a list'}
response = client.post('/predict', json=invalid_input)

assert response.status_code == 422, f'Expected 422, got {response.status_code}'

print('PASS — test_predict_wrong_data_type')
print(f'  Status : {response.status_code}')
print(f'  Detail : {response.json()["detail"][0]["msg"]}')

INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 422 Unprocessable Entity"


PASS — test_predict_wrong_data_type
  Status : 422
  Detail : Input should be a valid list


## 9. Test: Model Inference Failure (Mocked)

In [9]:
with patch('app.model.predict', side_effect=Exception('Model error')):
    response = client.post('/predict', json=valid_input_dict)

assert response.status_code == 500, f'Expected 500, got {response.status_code}'

print('PASS — test_predict_model_failure')
print(f'  Status : {response.status_code}')
print(f'  Detail : {response.json()}')

ERROR:app:Prediction error: Model error
INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 500 Internal Server Error"


PASS — test_predict_model_failure
  Status : 500
  Detail : {'detail': 'Model inference failed: Model error'}


## 10. Test: Performance (Latency < 500ms)

In [10]:
start_time = time.time()
response = client.post('/predict', json=valid_input_dict)
end_time = time.time()

latency_ms = (end_time - start_time) * 1000

assert response.status_code == 200, f'Expected 200, got {response.status_code}'
assert latency_ms < 500, f'Latency {latency_ms:.1f}ms exceeds 500ms threshold'

print('PASS — test_predict_performance')
print(f'  Status  : {response.status_code}')
print(f'  Latency : {latency_ms:.2f}ms')

INFO:httpx:HTTP Request: POST http://testserver/predict "HTTP/1.1 200 OK"


PASS — test_predict_performance
  Status  : 200
  Latency : 11.06ms


## 11. Run Full Suite via pytest

In [11]:
%%writefile test_app.py
import time
from unittest.mock import patch
import pytest
from fastapi.testclient import TestClient
from app import InputData, app

client = TestClient(app)

@pytest.fixture
def valid_input_data():
    return InputData(features=[1.0, 2.0, 3.0, 4.0])

@pytest.fixture
def valid_input_dict():
    return {'features': [1.0, 2.0, 3.0, 4.0]}

def test_predict_valid_input(valid_input_dict):
    response = client.post('/predict', json=valid_input_dict)
    assert response.status_code == 200
    assert 'prediction' in response.json()
    assert isinstance(response.json()['prediction'], (int, float))

def test_predict_invalid_input():
    response = client.post('/predict', json={'features': [1.0, 2.0]})
    assert response.status_code in [422, 500]

def test_predict_missing_required_field():
    response = client.post('/predict', json={})
    assert response.status_code == 422

def test_predict_wrong_data_type():
    response = client.post('/predict', json={'features': 'not a list'})
    assert response.status_code == 422

def test_predict_model_failure(valid_input_dict):
    with patch('app.model.predict', side_effect=Exception('Model error')):
        response = client.post('/predict', json=valid_input_dict)
    assert response.status_code == 500

def test_predict_performance(valid_input_dict):
    start = time.time()
    response = client.post('/predict', json=valid_input_dict)
    latency_ms = (time.time() - start) * 1000
    assert response.status_code == 200
    assert latency_ms < 500, f'Latency {latency_ms:.1f}ms exceeds 500ms'

Writing test_app.py


In [12]:
!pytest test_app.py -v

============================= test session starts =============================
platform win32 -- Python 3.13.9, pytest-8.4.2, pluggy-1.5.0 -- C:\Users\ryan\anaconda3\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\ryan
plugins: anyio-4.10.0
collecting ... collected 6 items

test_app.py::test_predict_valid_input PASSED                             [ 16%]
test_app.py::test_predict_invalid_input PASSED                           [ 33%]
test_app.py::test_predict_missing_required_field PASSED                  [ 50%]
test_app.py::test_predict_wrong_data_type PASSED                         [ 66%]
test_app.py::test_predict_model_failure PASSED                           [ 83%]
test_app.py::test_predict_performance PASSED                             [100%]

============================== 6 passed in 4.54s ==============================
